# Pipeline

> End-to-End Pipeline: PDF OCR, Markdown Heading Correction, and AI Image Descriptions

In [ ]:
#| default_exp pipeline

PDF to markdown pipeline with OCR, heading fixes, and AI-generated image descriptions.

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs, ocr_pdf
from mistocr.refine import add_img_descs, fix_hdgs
from pathlib import Path
from asyncio import Semaphore, gather, sleep
import tempfile
import os, json, shutil
import logging

In [ ]:
#| export
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.WARNING, format='%(name)s - %(levelname)s - %(message)s')
logger.setLevel(logging.INFO)

In [ ]:
#| export
@delegates(add_img_descs)
async def pdf_to_md(
    pdf_path:str,                   # Path to input PDF file
    dst:str,                        # Destination directory for output markdown
    ocr_dst:str=None,               # Optional OCR output directory
    model:str='claude-sonnet-4-5',  # Model to use for heading fixes and image descriptions
    add_img_desc:bool=True,         # Whether to add image descriptions
    progress:bool=True,             # Whether to show progress messages
    **kwargs
    ):
    "Convert PDF to markdown with OCR, fixed heading hierarchy, and optional image descriptions"
    "Convert PDF to markdown with OCR, fixed heading hierarchy, and optional image descriptions"
    cleanup = ocr_dst is None
    if cleanup: ocr_dst = tempfile.mkdtemp()
    n_steps = 3 if add_img_desc else 2
    if progress: logger.info(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_dir = ocr_pdf(pdf_path, ocr_dst)[0]
    if progress: logger.info(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: logger.info(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst != str(ocr_dir): shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if cleanup: shutil.rmtree(ocr_dst)
    if progress: logger.info("Done!")

In [ ]:
#| eval: false
await pdf_to_md('files/test/attention-is-all-you-need.pdf', 'files/test/md_test')

__main__ - INFO - Step 1/3: Running OCR on files/test/attention-is-all-you-need.pdf...


mistocr.core - INFO - Waiting for batch job 9dc9cc8c-f84d-430f-9435-45d167431f93 (initial status: QUEUED)


mistocr.core - DEBUG - Job 9dc9cc8c-f84d-430f-9435-45d167431f93 status: QUEUED (elapsed: 0s)


mistocr.core - DEBUG - Job 9dc9cc8c-f84d-430f-9435-45d167431f93 status: RUNNING (elapsed: 2s)


mistocr.core - DEBUG - Job 9dc9cc8c-f84d-430f-9435-45d167431f93 status: RUNNING (elapsed: 2s)


...

__main__ - INFO - Step 2/3: Fixing heading hierarchy...


__main__ - INFO - Step 3/3: Adding image descriptions...


Describing 7 images...


__main__ - INFO - Done!


Saved descriptions to /tmp/tmp1z_5vpmb/attention-is-all-you-need/img_descriptions.json
Adding descriptions to 15 pages...
Done! Enriched pages saved to files/test/md_test


In [ ]:
#| eval: false
!ls -R files/test/md_test

files/test/md_test:
img	    page_11.md	page_14.md  page_3.md  page_6.md  page_9.md
page_1.md   page_12.md	page_15.md  page_4.md  page_7.md
page_10.md  page_13.md	page_2.md   page_5.md  page_8.md

files/test/md_test/img:
img-0.jpeg  img-2.jpeg	img-4.jpeg  img-6.jpeg
img-1.jpeg  img-3.jpeg	img-5.jpeg


In [ ]:
#| eval: false
md = read_pgs('files/test/md_test', join=True)
print(md[5000:8000])


d of sequence-aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks *[34]*.

To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as *[17, 18]* and *[9]*.

## 3 Model Architecture ... page 2

Most competitive neural sequence transduction models have an encoder-decoder structure *[5, 2, 35]*. Here, the encoder maps an input sequence of symbol representations $(x_{1},...,x_{n})$ to a sequence of continuous representations $\mathbf{z}=(z_{1},...,z_{n})$. Given $\mathbf{z}$, the decoder then generates an output sequence $(y_{1},...,y_{m})$ of symbols one element at a time. At each step the model is auto-regressive *[10]*, consum